In [9]:
import frontmatter

with open('fm.md', 'r', encoding='utf-8') as f:
    post = frontmatter.load(f)

print(post.metadata['title'])
print(post.metadata['tags']) 

Getting Started with AI
['ai', 'machine-learning', 'tutorial']


In [10]:
print(post.content)

# Getting Started with AI

This is the main content of the document written in **Markdown**.

You can include code blocks, links, and other formatting here.


In [11]:
post.to_dict()

{'title': 'Getting Started with AI',
 'author': 'John Doe',
 'date': '2024-01-15',
 'tags': ['ai', 'machine-learning', 'tutorial'],
 'difficulty': 'beginner',
 'content': '# Getting Started with AI\n\nThis is the main content of the document written in **Markdown**.\n\nYou can include code blocks, links, and other formatting here.'}

In [4]:
import io
import zipfile
import requests
import frontmatter

from IPython.display import Markdown, display

In [74]:
url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'
resp = requests.get(url)

In [65]:
repository_data = []

# Create a ZipFile object from the downloaded content
zf = zipfile.ZipFile(io.BytesIO(resp.content))

In [34]:
for file_info in zf.infolist():
    filename = file_info.filename.lower()

    # Only process markdown files
    if not filename.endswith('.md'):
        continue

    # Read and parse each file
    with zf.open(file_info) as f_in:
        content = f_in.read()
        post = frontmatter.loads(content)
        data = post.to_dict()
        data['filename'] = filename
        repository_data.append(data)

zf.close()

In [23]:
repository_data

[{'description': 'Review and process open FAQ PRs',
  'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep

In [24]:
display(Markdown(repository_data[0]['content']))

Go through all open pull requests one by one. For each PR:

## 1. Show Details
- PR number and title
- Course and section (from PR body)
- Related issue number
- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential
- Full diff (use `gh pr diff <number>`)

## 2. Check Against These Rules

### Section Placement
- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`
- **Docker + Kestra** → still `module-2` (Kestra is primary topic)
- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`

### Relevance (Close If)
- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)
- Generic programming not tied to course content
- Already exists in DE zoomcamp when proposed for ML zoomcamp

### Duplicates (Check Before Creating)
- Search existing FAQs: `grep -r "keyword" _questions/`
- Same content should NOT exist in both DE and ML zoomcamp
- Enhance existing entries instead of duplicating

### Code Quality (Fix Before Merge)
- Python code must have proper indentation
- Check for file corruption (garbage characters at end)
- Code blocks should be syntactically correct

## 3. Ask User
After showing each PR, ask: "Merge, close, or needs changes?"

## 4. Actions
- **Merge**: `gh pr merge <number> --delete-branch --squash --subject "..." --body "..."`
- **Close**: `gh pr close <number> --comment "reason"`
- **Move section**: Checkout branch, `git mv` file, update sort_order, push, then merge
- **Fix content**: Checkout branch, edit file, commit, push, then merge

## 5. Sort Order Guidelines
- Find highest number in target section: `ls _questions/<course>/<section>/`
- Use next sequential number as sort_order

Get open PRs with:
`gh pr list --state open --json number,title,body,headRefName,baseRefName,url`

In [17]:
print(repository_data[1])

{'content': '# FAQ Bot Feedback - PR Review Corrections\n\n## 1. Wrong Section Placement\n\nKestra-related FAQs were incorrectly placed in `general` or `module-1` instead of `module-2` (workflow orchestration):\n\n| PR | Issue | Correction |\n|----|-------|------------|\n| #141 | Kestra IANA timezones | general → module-2, sort_order 20 |\n| #137 | Kestra stdout variables | general → module-2, sort_order 21 |\n| #135 | Kestra outputFiles visibility | general → module-2, sort_order 22 |\n| #118 | Kestra Docker socket | module-1 → module-2, sort_order 23 |\n\n**Rule**: Kestra questions belong in `module-2` (workflow orchestration), not `general` or `module-1`.\n\n---\n\n## 2. Not Relevant for Course (closed)\n\n| PR | Topic | Reason |\n|----|-------|--------|\n| #123 | Installing vim on Ubuntu | Basic Linux admin, outside course scope |\n| #116 | SQL LEFT JOIN returns NULL | Basic SQL concept, not course-specific |\n\n**Rule**: Fundamental tool/SQL concepts that aren\'t course-specific s

In [5]:
import io
import zipfile
import requests
import frontmatter
from IPython.display import Markdown, display

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md')
            or filename_lower.endswith('.mdx')):
            continue

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    zf.close()
    return repository_data

In [2]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1359
Evidently documents: 95


In [7]:
dtc_faq

[{'description': 'Review and process open FAQ PRs',
  'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep

In [6]:
display(Markdown(dtc_faq[0]['content']))

Go through all open pull requests one by one. For each PR:

## 1. Show Details
- PR number and title
- Course and section (from PR body)
- Related issue number
- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential
- Full diff (use `gh pr diff <number>`)

## 2. Check Against These Rules

### Section Placement
- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`
- **Docker + Kestra** → still `module-2` (Kestra is primary topic)
- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`

### Relevance (Close If)
- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)
- Generic programming not tied to course content
- Already exists in DE zoomcamp when proposed for ML zoomcamp

### Duplicates (Check Before Creating)
- Search existing FAQs: `grep -r "keyword" _questions/`
- Same content should NOT exist in both DE and ML zoomcamp
- Enhance existing entries instead of duplicating

### Code Quality (Fix Before Merge)
- Python code must have proper indentation
- Check for file corruption (garbage characters at end)
- Code blocks should be syntactically correct

## 3. Ask User
After showing each PR, ask: "Merge, close, or needs changes?"

## 4. Actions
- **Merge**: `gh pr merge <number> --delete-branch --squash --subject "..." --body "..."`
- **Close**: `gh pr close <number> --comment "reason"`
- **Move section**: Checkout branch, `git mv` file, update sort_order, push, then merge
- **Fix content**: Checkout branch, edit file, commit, push, then merge

## 5. Sort Order Guidelines
- Find highest number in target section: `ls _questions/<course>/<section>/`
- Use next sequential number as sort_order

Get open PRs with:
`gh pr list --state open --json number,title,body,headRefName,baseRefName,url`

In [7]:
evidently_docs

[{'title': 'Create Plant',
  'openapi': 'POST /plants',
  'content': '',
  'filename': 'docs-main/api-reference/endpoint/create.mdx'},
 {'title': 'Delete Plant',
  'openapi': 'DELETE /plants/{id}',
  'content': '',
  'filename': 'docs-main/api-reference/endpoint/delete.mdx'},
 {'title': 'Get Plants',
  'openapi': 'GET /plants',
  'content': '',
  'filename': 'docs-main/api-reference/endpoint/get.mdx'},
 {'title': 'Introduction',
  'description': 'Example section for showcasing API endpoints',
  'content': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"

In [8]:
display(Markdown(evidently_docs[3]['content']))

<Note>
  If you're not looking to build API reference documentation, you can delete
  this section by removing the api-reference folder.
</Note>

## Welcome

There are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.

<Card
  title="Plant Store Endpoints"
  icon="leaf"
  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"
>
  View the OpenAPI specification file
</Card>

## Authentication

All API endpoints are authenticated using Bearer tokens and picked up from the specification file.

```json
"security": [
  {
    "bearerAuth": []
  }
]
```